# 04 · La pregunta que decide el split — solución de referencia

Comparar filas nuevas de grupos conocidos con grupos completamente nuevos sin confundir las dos preguntas.

**Ideas que aparecen:** laboratorio 02; independencia y muestreo (cápsula 4). Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

Esta es una propuesta para comparar con tus ideas; no es la única solución posible.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## Datos hechos para revelar un error
Cada grupo ficticio tiene una firma numérica casi repetida y una
etiqueta aleatoria. Las filas de un grupo se parecen muchísimo.
Memorizar firmas permite reconocer grupos conocidos, pero no
predecir la etiqueta aleatoria de un grupo nuevo.

El objetivo de este laboratorio es **generalizar a grupos nuevos**.
Por eso un split por filas responde otra pregunta. No es que usar
grupos sea siempre la solución: si el uso real fueran nuevas filas
de esos mismos grupos, habría que diseñar esa evaluación.


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
df = pd.read_csv(DATOS / "desarrollo.csv")
columnas = [f"firma_{i}" for i in range(6)]
indices = np.arange(len(df))
aleatorio_train, aleatorio_val = train_test_split(indices, test_size=0.25, random_state=SEMILLA, stratify=df.clase)


## Zona de experimentación
`partir` debe devolver índices que no compartan grupos entre train
y validación. Usa `GroupShuffleSplit`, una sola partición, 25% de
grupos para validación y la semilla fijada. Después explica por qué
la accuracy por filas no estima el uso con grupos nuevos.
No busques semillas que eleven el resultado: eso usa validación para seleccionar.


In [ ]:
def partir(datos):
    separador = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEMILLA)
    return next(separador.split(datos, datos.clase, groups=datos.grupo))


## Comparación en las mismas condiciones
Para cada partición, ajusta un baseline mayoritario y k-NN con escalado solamente en ese entrenamiento. Compara también k-NN sobre el split por filas. Aquí la etiqueta de un grupo nuevo es aleatoria: subir el número no es el objetivo. La intersección de grupos revela qué situación está midiendo cada score.


In [ ]:
def evaluar_particion(tr, va):
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
    modelo.fit(df.iloc[tr][columnas], df.iloc[tr].clase)
    base = DummyClassifier(strategy="most_frequent").fit(df.iloc[tr][columnas], df.iloc[tr].clase)
    score = accuracy_score(df.iloc[va].clase, modelo.predict(df.iloc[va][columnas]))
    score_base = accuracy_score(df.iloc[va].clase, base.predict(df.iloc[va][columnas]))
    return modelo, float(score), float(score_base)

tr, va = partir(df)
modelo, score, score_base = evaluar_particion(tr, va)
_, score_filas, _ = evaluar_particion(aleatorio_train, aleatorio_val)
compartidos = sorted(set(df.iloc[tr].grupo) & set(df.iloc[va].grupo))
resultado = {"baseline": score_base, "validacion": score,
             "accuracy_por_filas": score_filas, "grupos_compartidos": compartidos}
assert not set(tr) & set(va) and len(tr) + len(va) == len(df)
# Una evidencia adicional: StandardScaler aprendió exclusivamente train.
assert np.allclose(modelo.steps[0][1].mean_, df.iloc[tr][columnas].mean())
print(resultado)


<details><summary>Idea · unidad</summary>La unidad de partición debe ser grupo, no fila, porque ese será el origen de los ejemplos futuros.</details>
<details><summary>Idea · API</summary>next(GroupShuffleSplit(...).split(X, y, groups=...)) devuelve índices de filas respetando grupos.</details>
<details><summary>Idea · evidencia</summary>Compara la intersección de conjuntos de grupos, además del score. Una métrica alta no certifica independencia.</details>


## Transferencia: decide antes de mirar el resultado
Evalúa los doce grupos reservados usando el modelo congelado. No debe haber ninguno en desarrollo. No esperes que las dos accuracy por grupos sean idénticas: son pocas unidades independientes. Como segunda situación, explica cómo separarías datos si quisieras predecir el mes siguiente de sensores ya conocidos (corte temporal, sin futuro en train).


In [ ]:
nuevo = pd.read_csv(DATOS / "transferencia.csv")
assert not set(df.grupo) & set(nuevo.grupo)
resultado["transferencia"] = float(accuracy_score(nuevo.clase, modelo.predict(nuevo[columnas])))
resultado["grupos_transferencia"] = int(nuevo.grupo.nunique())
fechas = pd.date_range("2026-01-01", periods=12, freq="MS")
corte = fechas[8]
pasado, futuro = fechas[fechas < corte], fechas[fechas >= corte]
assert pasado.max() < futuro.min()
resultado["corte_temporal_valido"] = True


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
colores_grupo = pd.Categorical(df.grupo).codes
ejes[0].scatter(df.iloc[tr].firma_0, df.iloc[tr].firma_1, c=colores_grupo[tr], cmap="tab20", vmin=0, vmax=df.grupo.nunique() - 1, marker="o", alpha=0.65, label="train")
ejes[0].scatter(df.iloc[va].firma_0, df.iloc[va].firma_1, c=colores_grupo[va], cmap="tab20", vmin=0, vmax=df.grupo.nunique() - 1, marker="x", s=75, label="validación")
ejes[0].set(title="Firmas casi repetidas: ¿dónde quedó cada grupo?", xlabel="firma 0", ylabel="firma 1")
ejes[0].legend()
ejes[1].bar(["por filas", "tu split", "grupos nuevos"], [resultado["accuracy_por_filas"], resultado["validacion"], resultado["transferencia"]], color=["#d3a05b", "#287da3", "#65b5af"])
ejes[1].set(title="Tres conjuntos, preguntas que hay que distinguir", ylabel="accuracy", ylim=(0, 1.05))
fig.tight_layout()
plt.show()


## Para seguir explorando
¿Qué futuro imagina cada split? Observa puntos casi repetidos de un grupo y busca dónde aparecen sus vecinos. ¿Cambiaría tu evaluación si el siguiente ejemplo perteneciera a un grupo conocido, a uno nuevo o al mes siguiente?


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert resultado['grupos_compartidos'] == []
assert resultado['accuracy_por_filas'] > resultado['validacion'] + 0.15
assert resultado['grupos_transferencia'] == 12 and resultado['corte_temporal_valido']


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '04_validacion', "version": 'solución de referencia',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'accuracy y solapamiento de grupos; una accuracy menor no es un fallo si cambia la pregunta', "split": 'datos de desarrollo: 24 grupos × 8 filas; transferencia: otros 12 grupos × 8 filas'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
